In [21]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from concurrent.futures import ThreadPoolExecutor


# Событие : (Смещение на каждой итерации, число итераций)
events_step_limit_dict = {'CommitCommentEvent': (400000, 20),
    'CreateEvent': (400000, 20),
    'DeleteEvent': (400000, 20),
    'DownloadEvent': (10000, 60),
    'Event': (10000, 60),
    'FollowEvent': (10000, 60),
    'ForkApplyEvent': (10000, 60),
    'ForkEvent': (400000, 20),
    'GistEvent': (400000, 20),
    'GollumEvent': (400000, 20),
    'IssueCommentEvent': (400000, 20),
    'IssuesEvent': (400000, 20),
    'MemberEvent': (400000, 20),
    'PublicEvent': (400000, 20),
    'PullRequestEvent': (400000, 20),
    'PullRequestReviewCommentEvent': (400000, 20),
    'PullRequestReviewEvent': (400000, 20),
    'PushEvent': (400000, 20),
    'ReleaseEvent': (400000, 20),
    'TeamAddEvent': (10000, 60),
    'WatchEvent': (400000, 20)}

column_names = [
    'file_time', 'event_type', 'actor_login', 'repo_name', 'created_at', 'updated_at',
    'action', 'comment_id', 'body', 'path', 'position', 'line', 'ref', 'ref_type',
    'creator_user_login', 'number', 'title', 'labels', 'state', 'locked',
    'assignee', 'assignees', 'comments', 'author_association', 'closed_at', 'merged_at',
    'merge_commit_sha', 'requested_reviewers', 'requested_teams', 'head_ref',
    'head_sha', 'base_ref', 'base_sha', 'merged', 'mergeable', 'rebaseable',
    'mergeable_state', 'merged_by', 'review_comments', 'maintainer_can_modify',
    'commits', 'additions', 'deletions', 'changed_files', 'diff_hunk', 'original_position',
    'commit_id', 'original_commit_id', 'push_size', 'push_distinct_size', 'member_login',
    'release_tag_name', 'release_name', 'review_state'
]

url = "https://play.clickhouse.com/play?user=play"

def process_event(event_type, value, offset=0, limit=2000):
    event_data_offset = []
    offset_step = value[0]
    steps = value[1]

    # Создаем новую вкладку для каждого запроса
    driver = webdriver.Chrome()
    
    for i in range(steps):
        driver.get(url)
        query_input = driver.find_element(By.ID, 'query')
        select_query = f'SELECT * FROM github_events WHERE event_type = \'{event_type}\' LIMIT {limit} OFFSET {offset}'
        query_input.send_keys(select_query)
        query_input.send_keys(Keys.CONTROL, Keys.RETURN)
        time.sleep(5)
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        table = soup.find('table', {'id': 'data-table'})

        if table:
            for row in table.find_all('tr')[1:]:
                columns = row.find_all('td')[1:]
                row_data = [col.text.strip() for col in columns]
                event_data_offset.append(row_data)
        else:
            break

        offset += offset_step

    driver.quit()  # Закрываем вкладки
    return event_data_offset

with ThreadPoolExecutor(max_workers=8) as executor:  # Указать количество потоков
    results = list(executor.map(process_event, events_step_limit_dict.keys(), events_step_limit_dict.values()))

data = pd.DataFrame()
# Объединяем результаты
for result in results:
    if result:
        data = pd.concat([data, pd.DataFrame(result)])


# Затем присваиваем имена столбцов к датафрейму
data.columns = column_names

# Сохраняем DataFrame в CSV-файл
data.to_csv('github_events_data.csv', index=False)


In [16]:
def get_steps(event_count):
    if event_count > 6000:
        return 400000, 20
    elif 6000 >= event_count > 5000:
        return 100000, 30
    else:
        return 10000, 60


df = pd.read_csv("github_events_data.csv")
events_step_limit_dict = dict(df.groupby(df["event_type"])["event_type"].count().apply(get_steps))
events_step_limit_dict

C:\Users\santiperro\AppData\Local\Temp\ipykernel_8528\4106185644.py:10: DtypeWarning: Columns (8,9,12,14,16,20,26,29,30,31,32,37,44,46,47,50,51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("github_events_data.csv")


{'CommitCommentEvent': (400000, 20),
 'CreateEvent': (400000, 20),
 'DeleteEvent': (400000, 20),
 'DownloadEvent': (10000, 60),
 'Event': (10000, 60),
 'FollowEvent': (10000, 60),
 'ForkApplyEvent': (10000, 60),
 'ForkEvent': (400000, 20),
 'GistEvent': (400000, 20),
 'GollumEvent': (400000, 20),
 'IssueCommentEvent': (400000, 20),
 'IssuesEvent': (400000, 20),
 'MemberEvent': (400000, 20),
 'PublicEvent': (400000, 20),
 'PullRequestEvent': (400000, 20),
 'PullRequestReviewCommentEvent': (400000, 20),
 'PullRequestReviewEvent': (400000, 20),
 'PushEvent': (400000, 20),
 'ReleaseEvent': (400000, 20),
 'TeamAddEvent': (10000, 60),
 'WatchEvent': (400000, 20)}